### Date of Data Collection: August 15, 2026 - 225 included items

In [1]:
import pandas as pd
import numpy as np
import requests
import dimcli
import time
import re
import json
from tqdm.notebook import tqdm
pd.set_option('display.max_colwidth', 500)

In [2]:
dimension_key = "[REDACTED]"  # Personal directory path removed for privacy
dimcli.login(key=dimension_key,
             endpoint="https://app.dimensions.ai")
dsl = dimcli.Dsl()

Dimcli - Dimensions API Client (v1.8)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.15
Method: manual login


In [3]:
working_dir = "[REDACTED]"  # Personal directory path removed for privacy
folder = f'{working_dir}/search2-225-included-articles'

In [4]:
f = f'{folder}/225-included-in-Dimensions.csv'

eppi_dimension = pd.read_csv(f)
print(eppi_dimension.shape)
print(eppi_dimension.isna().sum())
eppi_dimension.head(2)

(225, 5)
eppi_id               0
doi                  28
short_ti              0
title                 0
dimensions_pub_id    29
dtype: int64


,eppi_id,doi,short_ti,title,dimensions_pub_id
0,110102006,10.1016/j.cmi.2023.07.022,Zilberman (2023),Retracted publications in infectious diseases and clinical microbiology literature: an analysis using the retraction watch database.,pub.1162838205
1,110101930,10.1001/jamanetworkopen.2024.3173,Zhu (2024),Citations of microRNA Biomarker Articles That Were Retracted: A Systematic Review.,pub.1169955269


In [5]:
dimension_ids = eppi_dimension.loc[~eppi_dimension['dimensions_pub_id'].isna()]['dimensions_pub_id'].tolist()
print(len(dimension_ids))
dimension_ids[:2]

196


['pub.1162838205', 'pub.1169955269']

# Collect Direct Citations

In [6]:
search_df = eppi_dimension.loc[~eppi_dimension['dimensions_pub_id'].isna()]

direct_citations = {}

for i, rec in tqdm(search_df.iterrows(), total=search_df.shape[0]):

    dimension_id = rec['dimensions_pub_id']
    eppi_id = rec['eppi_id']
    
    direct_citations[eppi_id] = {'dimension_id': dimension_id, 
                                 'foward_citations': []}

    query = f'''
    search publications 
    where reference_ids in ["{dimension_id}"]
    return publications[id+doi+title+year]
    '''

    res = dsl.query_iterative(query, limit=1000, verbose=False)
    res_stats = res.stats
    res_stats = res_stats['total_count']
    
    if res_stats > 0:
        res_df = res.as_dataframe()
        n_retreived = res_df['id'].nunique()

        citing_dimension_ids = res_df['id'].tolist()
        direct_citations[eppi_id]['foward_citations'] = citing_dimension_ids

        if res_stats != n_retreived:
            print(f'EPPI ID: {eppi_id}, res_stats={res_stats:>3}, n_retreived={n_retreived:>3}')
    time.sleep(0.5)


  0%|          | 0/196 [00:00<?, ?it/s]

In [7]:
for eppi_id, cit_dict in direct_citations.items():
    if len(cit_dict['foward_citations']) == 0:
        print(f"No forward citations. EPPI ID: {eppi_id}, Dimensions ID: {cit_dict['dimension_id']}")

No forward citations. EPPI ID: 130735347, Dimensions ID: pub.1203625712
No forward citations. EPPI ID: 130735215, Dimensions ID: pub.1194922210
No forward citations. EPPI ID: 130735351, Dimensions ID: pub.1203023992
No forward citations. EPPI ID: 130742452, Dimensions ID: pub.1193639445
No forward citations. EPPI ID: 130734646, Dimensions ID: pub.1194455529
No forward citations. EPPI ID: 130734865, Dimensions ID: pub.1186766624
No forward citations. EPPI ID: 130735223, Dimensions ID: pub.1188197181
No forward citations. EPPI ID: 130735833, Dimensions ID: pub.1199927562
No forward citations. EPPI ID: 130734720, Dimensions ID: pub.1199787739
No forward citations. EPPI ID: 130735358, Dimensions ID: pub.1202735626
No forward citations. EPPI ID: 130734632, Dimensions ID: pub.1198765810
No forward citations. EPPI ID: 130736124, Dimensions ID: pub.1198012462
No forward citations. EPPI ID: 110102840, Dimensions ID: pub.1184055086
No forward citations. EPPI ID: 130734609, Dimensions ID: pub.120

In [8]:
direct_cit_df_fwd = [['eppi_id', 'dimensions_id', 'n_forward_cits', 'forward_cits']]

for eppi_id, cit_dict in direct_citations.items():
    dimensions_id = cit_dict['dimension_id']
    citing_ids = cit_dict['foward_citations']
    if len(citing_ids) == 0:
        n_forward_cits = 0
        forward_cits = np.nan
        row = [eppi_id, dimensions_id, n_forward_cits, forward_cits]
        direct_cit_df_fwd.append(row)
    else:
        n_forward_cits = len(citing_ids)
        forward_cits = '|'.join(citing_ids)
        row = [eppi_id, dimensions_id, n_forward_cits, forward_cits]
        direct_cit_df_fwd.append(row)

direct_cit_df_fwd = pd.DataFrame(direct_cit_df_fwd[1:], columns=direct_cit_df_fwd[0])
print(direct_cit_df_fwd.shape)
direct_cit_df_fwd.head()

(196, 4)


,eppi_id,dimensions_id,n_forward_cits,forward_cits
0,110102006,pub.1162838205,17,pub.1204913931|pub.1203347653|pub.1202011991|pub.1197675442|pub.1196687174|pub.1196589060|pub.1193352791|pub.1190562310|pub.1188186098|pub.1187383280|pub.1186080097|pub.1185467035|pub.1185372024|pub.1184945112|pub.1181124595|pub.1174142028|pub.1166582963
1,110101930,pub.1169955269,15,pub.1204913931|pub.1202011991|pub.1201063051|pub.1200324771|pub.1200289674|pub.1197666665|pub.1197165114|pub.1192598866|pub.1189237168|pub.1186525203|pub.1186080097|pub.1184945112|pub.1183600750|pub.1175377669|pub.1172115611
2,130735347,pub.1203625712,0,NaN
3,110104346,pub.1168449659,12,pub.1203641297|pub.1200823010|pub.1197848035|pub.1196096378|pub.1195870608|pub.1194918522|pub.1190991617|pub.1188962059|pub.1188820451|pub.1187272332|pub.1181161284|pub.1176096699
4,110105033,pub.1151889789,4,pub.1197848035|pub.1194918522|pub.1191623415|pub.1186132361


In [9]:
dimension_ids_str = json.dumps(dimension_ids)

query = f'''
search publications 
where id in {dimension_ids_str}
return publications[id+reference_ids]
'''

res = dsl.query_iterative(query, limit=1000, verbose=True)
res_stats = res.stats
res_stats = res_stats['total_count']

direct_cit_df_bwd = res.as_dataframe()
direct_cit_df_bwd = direct_cit_df_bwd.rename(columns={'id': 'dimensions_id', 
                                                      'reference_ids': 'backward_cits'})

direct_cit_df_bwd['n_backward_cits'] = direct_cit_df_bwd['backward_cits'].str.len()
direct_cit_df_bwd['backward_cits'] = direct_cit_df_bwd['backward_cits'].str.join('|')

direct_cit_df_bwd.loc[direct_cit_df_bwd['backward_cits'].isna(), 'n_backward_cits'] = 0
direct_cit_df_bwd['n_backward_cits'] = direct_cit_df_bwd['n_backward_cits'].astype(int)

print(direct_cit_df_bwd.shape)
direct_cit_df_bwd.head()

Starting iteration with limit=1000 skip=0 ...
0-196 / 196 (1.50s)
===
Records extracted: 196


(196, 3)


,dimensions_id,backward_cits,n_backward_cits
0,pub.1203625712,pub.1016940482|pub.1083878405|pub.1147360247|pub.1165151783|pub.1037347646|pub.1153157846|pub.1039257071|pub.1084030115|pub.1006106643|pub.1035913875|pub.1182017655|pub.1187512616|pub.1102828695|pub.1128978580|pub.1041784505|pub.1005866610|pub.1003766485|pub.1132618650|pub.1158448317|pub.1157399730|pub.1185383752|pub.1071063044,22
1,pub.1203311772,pub.1059436953|pub.1105578346|pub.1152864726|pub.1060406942|pub.1132949578|pub.1084810915|pub.1047164748|pub.1048595709|pub.1120764136|pub.1035913875|pub.1117981932|pub.1136204875|pub.1011730205|pub.1062731366|pub.1090371295|pub.1140001253|pub.1185025580|pub.1153020580|pub.1125675343|pub.1127498341|pub.1039625140|pub.1110308781|pub.1139691916|pub.1163840771|pub.1017763164|pub.1182281747|pub.1122561059|pub.1150821588|pub.1134167500|pub.1139997423|pub.1167083849|pub.1158321390|pub.1149986712|p...,49
2,pub.1203146058,pub.1187995646|pub.1189661055|pub.1187964516|pub.1187074984|pub.1199519517|pub.1026085167|pub.1165566951|pub.1006935082|pub.1030182761|pub.1131728768|pub.1166883970|pub.1140499085|pub.1016678991|pub.1035913875|pub.1010295273|pub.1132618650|pub.1141199930|pub.1175122676|pub.1135287850|pub.1049353983|pub.1169947556|pub.1196986324|pub.1149658753|pub.1134856664|pub.1194924711|pub.1168967512|pub.1175926025|pub.1191881655|pub.1136164675|pub.1121886815|pub.1182925498|pub.1131798051|pub.1062809102|p...,34
3,pub.1203023992,pub.1188186098|pub.1021234815|pub.1030891018|pub.1121469079|pub.1035913875|pub.1012818229|pub.1145167972|pub.1104164876|pub.1183955897|pub.1149068000|pub.1164234293|pub.1164816195|pub.1007876134|pub.1021073811|pub.1051486788|pub.1191440022|pub.1040754266|pub.1174076403|pub.1026982506|pub.1073708718|pub.1048342249,21
4,pub.1202826298,pub.1200823010|pub.1145744719|pub.1163808213|pub.1092171316|pub.1200908377|pub.1047277104|pub.1006930794|pub.1196986324|pub.1018044905|pub.1166883970|pub.1173551814|pub.1129791560|pub.1157189111|pub.1164231825|pub.1164978162|pub.1149073743|pub.1185383752|pub.1084030115|pub.1190562310|pub.1112943673|pub.1200016013|pub.1136204875|pub.1048342249|pub.1197440885|pub.1169897724|pub.1187512616|pub.1174044972|pub.1018250525|pub.1173664565|pub.1176014234|pub.1197631350|pub.1187814422|pub.1182744640|p...,47


In [10]:
direct_cit_df = direct_cit_df_fwd.merge(direct_cit_df_bwd)
print(direct_cit_df.shape)
direct_cit_df.head()

(196, 6)


,eppi_id,dimensions_id,n_forward_cits,forward_cits,backward_cits,n_backward_cits
0,110102006,pub.1162838205,17,pub.1204913931|pub.1203347653|pub.1202011991|pub.1197675442|pub.1196687174|pub.1196589060|pub.1193352791|pub.1190562310|pub.1188186098|pub.1187383280|pub.1186080097|pub.1185467035|pub.1185372024|pub.1184945112|pub.1181124595|pub.1174142028|pub.1166582963,pub.1143321492|pub.1143898164|pub.1130188988|pub.1125983355|pub.1150236067|pub.1151484788|pub.1139286088|pub.1154698915,8
1,110101930,pub.1169955269,15,pub.1204913931|pub.1202011991|pub.1201063051|pub.1200324771|pub.1200289674|pub.1197666665|pub.1197165114|pub.1192598866|pub.1189237168|pub.1186525203|pub.1186080097|pub.1184945112|pub.1183600750|pub.1175377669|pub.1172115611,pub.1140044294|pub.1148657817|pub.1153157846|pub.1158384443|pub.1107556956|pub.1125983355|pub.1125292026|pub.1138695503|pub.1120663862|pub.1107851165|pub.1123161331|pub.1130188988|pub.1025734006|pub.1136587866|pub.1104557132|pub.1165658416|pub.1166883970|pub.1092391317,18
2,130735347,pub.1203625712,0,NaN,pub.1016940482|pub.1083878405|pub.1147360247|pub.1165151783|pub.1037347646|pub.1153157846|pub.1039257071|pub.1084030115|pub.1006106643|pub.1035913875|pub.1182017655|pub.1187512616|pub.1102828695|pub.1128978580|pub.1041784505|pub.1005866610|pub.1003766485|pub.1132618650|pub.1158448317|pub.1157399730|pub.1185383752|pub.1071063044,22
3,110104346,pub.1168449659,12,pub.1203641297|pub.1200823010|pub.1197848035|pub.1196096378|pub.1195870608|pub.1194918522|pub.1190991617|pub.1188962059|pub.1188820451|pub.1187272332|pub.1181161284|pub.1176096699,pub.1120669062|pub.1102363322|pub.1111098983|pub.1032572062|pub.1029031628|pub.1024842295|pub.1004226160|pub.1106894346|pub.1137497865|pub.1016146084|pub.1110073618|pub.1152003858|pub.1117621271|pub.1145489241|pub.1115678492|pub.1135589144|pub.1062799338|pub.1043370747|pub.1044275888|pub.1060004529|pub.1100154793|pub.1131723398|pub.1112613086|pub.1139419578|pub.1043672210|pub.1036620532|pub.1127591106|pub.1027235151|pub.1064732709|pub.1047991970|pub.1053489114|pub.1005894197|pub.1132618650|p...,85
4,110105033,pub.1151889789,4,pub.1197848035|pub.1194918522|pub.1191623415|pub.1186132361,pub.1137497865|pub.1009569707|pub.1043672210,3


In [11]:
save2 = f'{folder}/direct_citations.tsv'
direct_cit_df.to_csv(save2, sep='\t', index=False)

## Collect Indirect Citations -- Forward Citations' Backward Citations
- For each article that cite the included study, collect its references

In [12]:
unique_forward_cit_ids = []

foward_citations_lists = direct_cit_df['forward_cits'].tolist()
n_zero_foward = 0
for foward_citations_list in foward_citations_lists:
    if isinstance(foward_citations_list, str):
        foward_citations_list = foward_citations_list.split('|')
        unique_forward_cit_ids += foward_citations_list
    else:
        n_zero_foward += 1
    
unique_forward_cit_ids = list(set(unique_forward_cit_ids))
print(len(unique_forward_cit_ids))
print(n_zero_foward)

1900
31


In [13]:
forward_cit_id_batches = np.array_split(unique_forward_cit_ids, 5)
len(forward_cit_id_batches[0]), len(forward_cit_id_batches[-1])

(380, 380)

In [14]:
indirect_cit_backward = []

for id_batch in forward_cit_id_batches:
    
    id_batch = id_batch.tolist()
    dimension_ids_str = json.dumps(id_batch)

    query = f'''
    search publications 
    where id in {dimension_ids_str}
    return publications[id+reference_ids]
    '''

    res = dsl.query_iterative(query, limit=1000, verbose=True)
    res_stats = res.stats
    res_stats = res_stats['total_count']

    res_df = res.as_dataframe()
    indirect_cit_backward.append(res_df)
    
    time.sleep(1)

indirect_cit_backward = pd.concat(indirect_cit_backward)
indirect_cit_backward = indirect_cit_backward.rename(columns={'id': 'forward_cit_dimension_id', 
                                                              'reference_ids': 'forward_cit_refs'})

indirect_cit_backward['n_forward_cit_refs'] = indirect_cit_backward['forward_cit_refs'].str.len()
indirect_cit_backward['forward_cit_refs'] = indirect_cit_backward['forward_cit_refs'].str.join('|')

print(indirect_cit_backward.shape, len(unique_forward_cit_ids))
indirect_cit_backward

Starting iteration with limit=1000 skip=0 ...
0-380 / 380 (4.81s)
===
Records extracted: 380
Starting iteration with limit=1000 skip=0 ...
0-380 / 380 (0.84s)
===
Records extracted: 380
Starting iteration with limit=1000 skip=0 ...
0-380 / 380 (1.70s)
===
Records extracted: 380
Starting iteration with limit=1000 skip=0 ...
0-380 / 380 (0.78s)
===
Records extracted: 380
Starting iteration with limit=1000 skip=0 ...
0-380 / 380 (0.81s)
===
Records extracted: 380


(1900, 3) 1900


,forward_cit_dimension_id,forward_cit_refs,n_forward_cit_refs
0,pub.1204382070,pub.1149068000|pub.1148284482|pub.1107861413|pub.1160792209|pub.1136754899|pub.1132151078|pub.1035913875|pub.1121827855|pub.1026451650|pub.1150571139|pub.1111083337|pub.1187074984|pub.1062781821|pub.1187964516|pub.1166883970|pub.1117291009|pub.1131595529|pub.1192669888|pub.1159735337|pub.1147566445|pub.1200485202|pub.1164231825|pub.1155707664,23
1,pub.1204209654,pub.1026590392|pub.1007613950|pub.1015617264|pub.1034661168|pub.1036328193|pub.1021897019|pub.1014471288|pub.1030028363|pub.1073712408|pub.1025183064|pub.1046626487|pub.1083804470|pub.1022885297|pub.1030080199|pub.1027408499|pub.1091278374|pub.1014576523|pub.1004225768|pub.1041290378|pub.1020300519|pub.1100342751|pub.1092573829|pub.1046995974,23
2,pub.1204182718,pub.1159958484|pub.1182846012|pub.1187964516|pub.1159735337|pub.1148221236|pub.1121827855|pub.1149068000|pub.1186933639,8
3,pub.1203871790,pub.1004779091|pub.1125904589|pub.1174023958|pub.1157879891|pub.1185372024|pub.1084030115|pub.1136368161|pub.1011930260|pub.1137488323|pub.1107896157|pub.1035913875|pub.1003571146|pub.1029926631|pub.1071063044|pub.1112987045|pub.1182281747|pub.1018063518|pub.1140499085|pub.1166883970|pub.1048342249|pub.1149073743|pub.1146805626|pub.1092536547|pub.1006277806|pub.1085062011|pub.1171367266|pub.1148433654|pub.1131737983|pub.1138277520|pub.1071063564|pub.1107318347|pub.1136164675|pub.1145209041|p...,34
4,pub.1203672890,pub.1151118650|pub.1100459294|pub.1035913875|pub.1196856658|pub.1130863323|pub.1141199930|pub.1022026507|pub.1037810965|pub.1041476956|pub.1107896157|pub.1137812626|pub.1018250525,12
...,...,...,...
375,pub.1009569707,pub.1047991970|pub.1044275888|pub.1019406465|pub.1037071040|pub.1018250525|pub.1062533850|pub.1001214533|pub.1023743967|pub.1051486788|pub.1031714110|pub.1016089443|pub.1074545715|pub.1049576378|pub.1071063474|pub.1079210410|pub.1077279338|pub.1024632875|pub.1005309576|pub.1008136513,19
376,pub.1005780705,pub.1075251550|pub.1046111483|pub.1047585028|pub.1052914763|pub.1069324617|pub.1041015426|pub.1033302851|pub.1036820722|pub.1043585451|pub.1043189107|pub.1022275528|pub.1024639728|pub.1025755064|pub.1005309209|pub.1062790299|pub.1024665019|pub.1064537168|pub.1045364789|pub.1035616315|pub.1062778767|pub.1009018107|pub.1054167307|pub.1034971106|pub.1076583434|pub.1017541403|pub.1017619963|pub.1062778747,27
377,pub.1004491854,pub.1062640896|pub.1054151048|pub.1062458806|pub.1062557439|pub.1054151057|pub.1054151055|pub.1062647540|pub.1070142332|pub.1074939855|pub.1055345772|pub.1009766587|pub.1074931796|pub.1010606947|pub.1056527254|pub.1054155784|pub.1037474081|pub.1054151046|pub.1055358566|pub.1055354685|pub.1020030585|pub.1055355504|pub.1062544028|pub.1024577511|pub.1062559276|pub.1074930445|pub.1016314397|pub.1062655540|pub.1074930473|pub.1013793596|pub.1005309576|pub.1052966239|pub.1054151037|pub.1018151357|p...,61
378,pub.1003723106,pub.1033468476|pub.1057119917|pub.1013080253|pub.1027290047|pub.1046352248|pub.1027136308|pub.1012583575|pub.1030818182|pub.1051002544|pub.1078307042|pub.1013303411|pub.1018605373|pub.1033981724|pub.1012053609|pub.1034313645|pub.1031452890|pub.1033230230|pub.1048489532|pub.1019041934|pub.1019413762|pub.1029777370|pub.1040605198|pub.1004230881|pub.1030744380|pub.1037857864|pub.1016051282|pub.1021342297|pub.1038796454|pub.1016846997|pub.1079531912|pub.1018146509|pub.1026265356|pub.1077535320|p...,60


In [15]:
indirect_cit_backward.loc[indirect_cit_backward['n_forward_cit_refs']==0]

,forward_cit_dimension_id,forward_cit_refs,n_forward_cit_refs


In [16]:
indirect_cit_backward.head(2)

,forward_cit_dimension_id,forward_cit_refs,n_forward_cit_refs
0,pub.1204382070,pub.1149068000|pub.1148284482|pub.1107861413|pub.1160792209|pub.1136754899|pub.1132151078|pub.1035913875|pub.1121827855|pub.1026451650|pub.1150571139|pub.1111083337|pub.1187074984|pub.1062781821|pub.1187964516|pub.1166883970|pub.1117291009|pub.1131595529|pub.1192669888|pub.1159735337|pub.1147566445|pub.1200485202|pub.1164231825|pub.1155707664,23
1,pub.1204209654,pub.1026590392|pub.1007613950|pub.1015617264|pub.1034661168|pub.1036328193|pub.1021897019|pub.1014471288|pub.1030028363|pub.1073712408|pub.1025183064|pub.1046626487|pub.1083804470|pub.1022885297|pub.1030080199|pub.1027408499|pub.1091278374|pub.1014576523|pub.1004225768|pub.1041290378|pub.1020300519|pub.1100342751|pub.1092573829|pub.1046995974,23


In [17]:
print(indirect_cit_backward.shape, len(unique_forward_cit_ids))

(1900, 3) 1900


In [18]:
save2 = f'{folder}/indirect_citations_forward_refs.tsv'
indirect_cit_backward.to_csv(save2, sep='\t', index=False)

## Collect Indirect Citations -- Backward Citations' Forward Citations
- For each article cited by the included study, collect its citing articles

In [19]:
def build_network_dict(seed, pubs_list):
    network={x:[] for x in seed} # seed a dictionary
    for pub in pubs_list:
        for key in network:
            if pub.get('reference_ids') and key in pub['reference_ids']:
                network[key].append(pub['id'])
    return network

In [20]:
unique_backward_cit_ids = []

backward_citations_lists = direct_cit_df['backward_cits'].tolist()
n_zero_backward = 0
for backward_citations_list in backward_citations_lists:
    if isinstance(backward_citations_list, str):
        backward_citations_list = backward_citations_list.split('|')
        unique_backward_cit_ids += backward_citations_list
    else:
        n_zero_backward += 1
    
unique_backward_cit_ids = list(set(unique_backward_cit_ids))
print(len(unique_backward_cit_ids))
print(n_zero_backward)

3000
13


In [21]:
direct_cit_df.loc[direct_cit_df['n_backward_cits']==0].shape

(13, 6)

In [23]:
ref_foward_citations = {}

for ref_dim_id in tqdm(unique_backward_cit_ids, total=len(unique_backward_cit_ids)):
    
    ref_foward_citations[ref_dim_id] = {'foward_citations': []}

    query = f'''
    search publications 
    where reference_ids = "{ref_dim_id}"
    return publications[id]
    '''

    res = dsl.query_iterative(query, limit=1000, verbose=False)
    res_stats = res.stats
    res_stats = res_stats['total_count']
    
    if res_stats > 0:
        res_df = res.as_dataframe()
        n_retreived = res_df['id'].nunique()
        
        if res_stats == n_retreived:
            citing_dimension_ids = res_df['id'].tolist()
            ref_foward_citations[ref_dim_id]['foward_citations'] = citing_dimension_ids

        if res_stats != n_retreived:
            print(f'ref_dim_id: {ref_dim_id}, res_stats={res_stats:>3}, n_retreived={n_retreived:>3}')
            print(f'Breaking down by year...')
            
            citing_dimension_ids = []
            n_retreived2 = 0
            
            year_query = f"""
            search publications
            where reference_ids = "{ref_dim_id}"
            return year limit 1000
            """
            
            year_counts = (
                dsl.query(year_query, verbose=False)
                   .as_dataframe()
                   .rename(columns={"id": "year"})
            )

            year_counts["year"] = year_counts["year"].astype(int)
            year_counts = year_counts.sort_values(by=['year'])
            
            for yr, yr_count in year_counts[['year', 'count']].values.tolist():
                query2 = f"""
                search publications
                where reference_ids = "{ref_dim_id}"
                  and year = {yr}
                return publications[id]
                """
                
                res2 = dsl.query_iterative(query2, limit=1000, verbose=False)
                res2_df = res2.as_dataframe()
                n_retreived_yr = res2_df['id'].nunique()
                
                if yr_count == n_retreived_yr:
                    n_retreived2 += n_retreived_yr
                    citing_dimension_ids += res2_df['id'].tolist()
                else:
                    print(f'ref_dim_id: {ref_dim_id}, yr={yr}, yr_count={yr_count}, n_retreived_yr={n_retreived_yr}')
                    n_retreived2 += n_retreived_yr
                    citing_dimension_ids += res2_df['id'].tolist()
            
            ref_foward_citations[ref_dim_id]['foward_citations'] = citing_dimension_ids
            
    time.sleep(0.5)


  0%|          | 0/3000 [00:00<?, ?it/s]

ref_dim_id: pub.1028525626, res_stats=53586, n_retreived=50000
Breaking down by year...
ref_dim_id: pub.1069974986, res_stats=70856, n_retreived=50000
Breaking down by year...
ref_dim_id: pub.1037575666, res_stats=65759, n_retreived=50000
Breaking down by year...
ref_dim_id: pub.1136754899, res_stats=109018, n_retreived=50000
Breaking down by year...


In [24]:
len(ref_foward_citations)

3000

In [25]:
ref_foward_citations_df = [['backward_cit_dimension_id', 
                            'backward_cit_citing_articles', 'n_backward_cit_citing_articles']]

for ref, ref_dict in ref_foward_citations.items():
    backward_cit_citing_articles = ref_dict['foward_citations']
    n_backward_cit_citing_articles = len(backward_cit_citing_articles)
    
    backward_cit_citing_articles = '|'.join(backward_cit_citing_articles)
    
    row = [ref, backward_cit_citing_articles, n_backward_cit_citing_articles]
    ref_foward_citations_df.append(row)
    
ref_foward_citations_df = pd.DataFrame(ref_foward_citations_df[1:], columns=ref_foward_citations_df[0])
print(ref_foward_citations_df.shape)
ref_foward_citations_df.head(2)

(3000, 3)


,backward_cit_dimension_id,backward_cit_citing_articles,n_backward_cit_citing_articles
0,pub.1091634367,pub.1203897664|pub.1203882268|pub.1202606800|pub.1200554390|pub.1199425080|pub.1198774155|pub.1197626039|pub.1195944924|pub.1193968904|pub.1191483106|pub.1189652686|pub.1188765985|pub.1188765969|pub.1188659889|pub.1187961501|pub.1187303589|pub.1185794484|pub.1185527747|pub.1185400413|pub.1182396927|pub.1182082555|pub.1182071664|pub.1181935479|pub.1175917153|pub.1175207416|pub.1174955115|pub.1171744450|pub.1171684045|pub.1171317997|pub.1170927438|pub.1169923242|pub.1169002113|pub.1168944885|p...,161
1,pub.1022346256,pub.1204634543|pub.1203109228|pub.1198164047|pub.1190151631|pub.1184150445|pub.1183760762|pub.1182315073|pub.1173232043|pub.1170397241|pub.1160162448|pub.1157965340|pub.1151743026|pub.1145208764|pub.1145028874|pub.1142333271|pub.1141844112|pub.1138575657|pub.1138352012|pub.1137100411|pub.1136510746|pub.1129072219|pub.1128730057|pub.1126088261|pub.1119949211|pub.1117710929|pub.1116663417|pub.1111931728|pub.1111829796|pub.1105970606|pub.1105932736|pub.1101748562|pub.1099693949|pub.1092879432|p...,140


In [26]:
ref_foward_citations_df.loc[ref_foward_citations_df['backward_cit_dimension_id']=='pub.1037575666']

,backward_cit_dimension_id,backward_cit_citing_articles,n_backward_cit_citing_articles
2796,pub.1037575666,pub.1033011775|pub.1012876427|pub.1054036268|pub.1052451805|pub.1049012697|pub.1046350879|pub.1043673086|pub.1042340916|pub.1033759600|pub.1033344033|pub.1024994299|pub.1024783728|pub.1023859332|pub.1022806278|pub.1022531365|pub.1015745369|pub.1015174578|pub.1013205257|pub.1012246389|pub.1011567860|pub.1003861124|pub.1113788167|pub.1078924858|pub.1073711602|pub.1070837491|pub.1068737014|pub.1064291882|pub.1063726306|pub.1053793641|pub.1053232935|pub.1052697811|pub.1051821989|pub.1050683954|p...,65752


In [27]:
folder

'/Users/estherhsiao/Documents/UIUC/0X_Jodi_lab/post-retraction citation - lit review/update_search/search2-225-included-articles'

In [28]:
save2 = f'{folder}/indirect_citations_backward_citing.tsv'
ref_foward_citations_df.to_csv(save2, sep='\t', index=False)